# 10.11 Score-based models and SDEs

Score models learn the direction back toward data after noise has blurred the distribution.

The score points toward higher density at each noise level, becoming the vector field used by diffusion and classifier-free guidance.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

SEED = 1008
rng = np.random.default_rng(SEED)


def standardize_features(X):
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True) + 1e-6
    return (X - mu) / sd


def pca_features(X, n_components=8):
    X = standardize_features(X)
    k = min(n_components, X.shape[1], max(1, X.shape[0] - 1))
    if k == X.shape[1]:
        return X
    pca = PCA(n_components=k, random_state=SEED)
    return pca.fit_transform(X)


def make_f9_ladder(seed=SEED):
    local = np.random.default_rng(seed)
    rungs = []

    x1 = local.normal(0.0, 1.0, size=(240, 1))
    y1 = (x1[:, 0] > 0).astype(int)
    rungs.append({"name": "D1 1-D Gaussian", "X": x1, "y": y1, "kind": "points"})

    x2, y2 = make_moons(n_samples=320, noise=0.08, random_state=seed + 2)
    rungs.append({"name": "D2 2-D moons", "X": x2, "y": y2, "kind": "points"})

    means = np.array([[-2.0, -1.0], [1.8, -0.6], [0.2, 2.0]])
    covs = np.array([[[0.20, 0.05], [0.05, 0.45]], [[0.45, -0.18], [-0.18, 0.25]], [[0.25, 0.0], [0.0, 0.25]]])
    parts = []
    labels = []
    for k, (mean, cov) in enumerate(zip(means, covs)):
        pts = local.multivariate_normal(mean, cov, size=120)
        parts.append(pts)
        labels.append(np.full(120, k))
    x3 = np.vstack(parts)
    y3 = np.concatenate(labels)
    rungs.append({"name": "D3 three-component mixture", "X": x3, "y": y3, "kind": "points"})

    digits = load_digits()
    x4 = digits.data.astype(float) / 16.0
    y4 = digits.target.astype(int)
    rungs.append({"name": "D4 sklearn digits", "X": x4[:900], "y": y4[:900], "kind": "image", "image_shape": (8, 8)})

    hard = x4.copy()
    hard = hard + local.normal(0.0, 0.18, size=hard.shape)
    hard = np.clip(hard, 0.0, 1.0)
    hard[:, ::3] = 0.65 * hard[:, ::3] + 0.35 * local.random(size=hard[:, ::3].shape)
    hard = np.roll(hard.reshape(-1, 8, 8), shift=1, axis=2).reshape(-1, 64)
    rungs.append({"name": "D5 harder real digits (shift+noise)", "X": hard[:900], "y": y4[:900], "kind": "image", "image_shape": (8, 8)})

    return rungs


def generated_baseline(rung, seed=SEED):
    local = np.random.default_rng(seed)
    X = np.asarray(rung["X"], dtype=float)
    if X.shape[1] == 1:
        return local.normal(X.mean(axis=0), X.std(axis=0) + 1e-6, size=X.shape)
    idx = local.choice(len(X), size=len(X), replace=True)
    boot = X[idx].copy()
    noise = local.normal(0.0, 0.10 * (X.std(axis=0) + 1e-6), size=boot.shape)
    return boot + noise


def feature_matrix(X):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    if X.shape[1] > 10:
        return pca_features(X, n_components=10)
    return standardize_features(X)


def sqrtm_psd(A):
    vals, vecs = np.linalg.eigh((A + A.T) / 2.0)
    vals = np.clip(vals, 0.0, None)
    return (vecs * np.sqrt(vals)) @ vecs.T


def fid_distance(real, gen):
    R = feature_matrix(real)
    G = feature_matrix(gen)
    mr = R.mean(axis=0)
    mg = G.mean(axis=0)
    cr = np.cov(R, rowvar=False) + np.eye(R.shape[1]) * 1e-6
    cg = np.cov(G, rowvar=False) + np.eye(G.shape[1]) * 1e-6
    middle = sqrtm_psd(sqrtm_psd(cr) @ cg @ sqrtm_psd(cr))
    return float(np.sum((mr - mg) ** 2) + np.trace(cr + cg - 2.0 * middle))


def two_sample_distance(real, gen):
    return fid_distance(real, gen)


def gaussian_nll(real, gen=None):
    X = feature_matrix(real)
    mu = X.mean(axis=0, keepdims=True)
    var = X.var(axis=0, keepdims=True) + 1e-4
    logp = -0.5 * (((X - mu) ** 2) / var + np.log(2.0 * np.pi * var))
    return float(-logp.sum(axis=1).mean())


def show_artifact(ax, X, rung, title):
    X = np.asarray(X)
    if rung.get("kind") == "image":
        img = X[0].reshape(rung.get("image_shape", (8, 8)))
        ax.imshow(img, cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    elif X.shape[1] == 1:
        ax.hist(X[:, 0], bins=24, color="steelblue", alpha=0.8)
    else:
        ax.scatter(X[:, 0], X[:, 1], s=8, c=rung.get("y"), cmap="tab10", alpha=0.75)
        ax.set_xticks([])
        ax.set_yticks([])
    ax.set_title(title, fontsize=9)


def plot_generated_panels(rungs, generated, metrics, ylabel):
    fig, axes = plt.subplots(2, len(rungs), figsize=(3.0 * len(rungs), 5.2))
    for j, rung in enumerate(rungs):
        show_artifact(axes[0, j], rung["X"], rung, "real " + rung["name"][:16])
        show_artifact(axes[1, j], generated[j], rung, "generated")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(np.arange(1, len(metrics) + 1), metrics, marker="o")
    ax.set_xlabel("rung D1 to D5")
    ax.set_ylabel(ylabel)
    ax.set_title("Metric curve across the F9 ladder")
    ax.grid(True, alpha=0.3)
    plt.show()


def preview_ladder(rungs):
    for i, rung in enumerate(rungs, start=1):
        X = rung["X"]
        y = rung.get("y")
        labels = "none" if y is None else np.unique(y)[:10].tolist()
        print(f"D{i}: {rung['name']}: X={X.shape}, labels/classes={labels}")


def kde_score(points, query, sigma):
    points = np.asarray(points, dtype=float)
    query = np.asarray(query, dtype=float)
    diff = points[None, :, :] - query[:, None, :]
    weights = np.exp(-0.5 * np.sum(diff ** 2, axis=2) / (sigma ** 2))
    denom = weights.sum(axis=1, keepdims=True) + 1e-9
    return (weights[:, :, None] * diff).sum(axis=1) / denom / (sigma ** 2)


def predictor_corrector_sample(points, n, sigmas, step_size=0.015, seed=SEED):
    local = np.random.default_rng(seed)
    F = feature_matrix(points)
    x = local.normal(0.0, sigmas[0], size=(n, F.shape[1]))
    for sigma in sigmas:
        for _ in range(12):
            score = kde_score(F, x, sigma)
            x = x + step_size * score + np.sqrt(2.0 * step_size) * local.normal(size=x.shape)
    return x


## The concept, built once: the Gaussian score
A score model learns $$s_	heta(x,t)pprox 
abla_x\log p_t(x).$$ For $p_t(x)=\mathcal{N}(0,\sigma^2)$, the exact score is $-x/\sigma^2$.

In [ ]:
def gaussian_score_and_langevin(x, sigma, step_size=0.05):
    score = -np.asarray(x, dtype=float) / (sigma ** 2)
    denoised = np.asarray(x, dtype=float) + step_size * score
    return score, denoised

score_left, denoised_left = gaussian_score_and_langevin(np.array([-1.0]), 0.5)
score_right, denoised_right = gaussian_score_and_langevin(np.array([1.0]), 0.5)
print(score_left[0], score_right[0])
assert round(float(score_left[0]), 3) == 4.000
assert round(float(score_right[0]), 3) == -4.000

The same idea becomes a tiny SDE sampler: start from broad Gaussian noise, estimate a KDE score at each noise level, and take small predictor-corrector steps toward the data manifold.

In [ ]:
def score_sample_for_rung(X, seed=SEED):
    sigmas = np.array([1.2, 0.7, 0.35, 0.18])
    samples = predictor_corrector_sample(X, len(X), sigmas, seed=seed)
    return samples

## The dataset ladder
We use the same F9 ladder inline in every notebook: 1-D Gaussian, 2-D moons, a three-component mixture, real sklearn digits, and a harder no-download real-digits rung with shift plus noise. This keeps the CPU path deterministic while increasing sample complexity.

In [ ]:
rungs = make_f9_ladder()
preview_ladder(rungs)

fig, axes = plt.subplots(1, len(rungs), figsize=(14, 2.6))
for ax, rung in zip(axes, rungs):
    show_artifact(ax, rung["X"], rung, rung["name"])
plt.tight_layout()
plt.show()

In [ ]:
metrics = []
generated = []
for rung in rungs:
    samples = score_sample_for_rung(rung["X"])
    generated.append(samples)
    metric = two_sample_distance(feature_matrix(rung["X"]), samples)
    metrics.append(metric)
    print(f"{rung['name']:<34} score 2-sample distance={metric:8.4f}")

In [ ]:
plot_generated_panels(rungs, generated, metrics, "2-sample distance")

## Pitfall on D5: one noise level and too-large steps
A single score field does not provide the path from pure noise back to data, and large discretization steps can overshoot the score field.

In [ ]:
d5 = rungs[-1]
one_level = predictor_corrector_sample(d5["X"], 300, np.array([0.18]), step_size=0.20, seed=SEED + 3)
ladder = predictor_corrector_sample(d5["X"], 300, np.array([1.2, 0.7, 0.35, 0.18]), step_size=0.015, seed=SEED + 4)
bad = two_sample_distance(feature_matrix(d5["X"])[:300], one_level)
good = two_sample_distance(feature_matrix(d5["X"])[:300], ladder)
print(f"one-level/large-step distance={bad:.3f}")
print(f"noise-ladder/small-step distance={good:.3f}")

## Evaluate it + Practice
- Report `2-sample distance` beside a no-skill baseline that resamples training examples with small Gaussian jitter.
- Cheap sanity check: generated panels should have the same support and rough diversity as the real panels.
- Ablation: train/sample with one noise level or make the SDE step size too large; the metric should get worse.
- Failure signal: D5 can look plausible in one panel while the summary curve exposes collapse or oversmoothing.
- Re-run with a different seed only after the structural logic is correct.

Practice prompts:
1. Change the D3 mixture weights and predict which metric moves most.
2. Add a bootstrap interval for D5 before trusting a small metric gap.
3. Replace PCA features with raw features on D4/D5 and explain the tradeoff.